In [3]:
import numpy as np
import pandas as pd
from scipy import signal
from scipy.io import wavfile
from scipy.fft import fftshift
import matplotlib.pyplot as plt
import librosa
from scipy.signal import hilbert, butter, lfilter
import os
import textgrid
import sys
import shutil
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import precision_recall_curve

In [4]:
def get_files(folder, id_list):
    files = os.listdir(folder)
    list_files = []
    for id_indiv in id_list:
        yes_files = [f for f in files if "YH_"+str(id_indiv)+"_" in f]
        list_files.extend(yes_files)
    return list_files

In [5]:
# I got the code from here https://scipy-cookbook.readthedocs.io/items/ButterworthBandpass.html 
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band') # from scipy.signal
    return b, a


def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    #plot_filter(b,a)
    y = lfilter(b, a, data)
    return y

In [6]:
def energy(signal):
    origine = np.sqrt(signal@signal.T/signal.shape[0]) #np.mean(np.abs(signal)) #
    db = librosa.amplitude_to_db(origine)
    return db

In [7]:
def values_db (folder_db,files, exclude_far=False):
    if exclude_far:
        files = [file for file in files if "100m" not in file]
        files = [file for file in files if "150m" not in file]
        files = [file for file in files if "200m" not in file]

    results = []

    for song_file in files:
        samplerate, data = wavfile.read(folder_db+song_file)
        data = data.astype(float)
        data = butter_bandpass_filter(data, 4000, 9000, samplerate, order=6)
        
        if data.shape[0] != 0.5*samplerate:
            print("Not a 500ms clip!")

        results.append(energy(data))

    return results

In [8]:
def baseline(signal, srate, reference=40):

    assert signal.shape[0] >= int(srate*0.5)

    filtered_signal = butter_bandpass_filter(signal, 4000, 9000, srate, order=6)
    
    db = energy(filtered_signal)

    return db>=reference

In [9]:
def predictions_baseline (folder_db,files, df_threshold, exclude_far=False):
    if exclude_far:
        files = [file for file in files if "100m" not in file]
        files = [file for file in files if "150m" not in file]
        files = [file for file in files if "200m" not in file]

    results = []

    for song_file in files:
        samplerate, data = wavfile.read(folder_db+song_file)
        data = data.astype(float)
        data = butter_bandpass_filter(data, 4000, 9000, samplerate, order=6)
        
        if data.shape[0] != 0.5*samplerate:
            print("Not a 500ms clip!")

        results.append(baseline(data,samplerate, reference=df_threshold))

    return results

In [10]:
def get_ids (files):
    ids = [f.split("_")[-3] for f in files]
    return np.unique(ids)

In [11]:
def negative_folds(folder_negative):
    files = os.listdir(folder_negative)
    files = [f for f in files if ".wav"]
    y = [0 for f in files]

    X_train, X_test, y_train, y_test = train_test_split(files, y, test_size=0.2, random_state=1)

    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=1)

    return X_train, X_val, X_test

In [12]:
def get_slice(list, start, end):
    return list[int(start*len(list)):int(end*len(list))]

In [13]:
def postive_rate(check):
    if len(check)==0:
        return 0
    return len([b for b in check if b==True])/len(check)

In [14]:
def results (folder, files, threshold, exclude=False):

    all_p = predictions_baseline (folder, files, df_threshold=threshold, exclude_far=exclude)

    pr_all = postive_rate(all_p)
    all_rates = np.array([pr_all, 1-pr_all])

    return np.round(all_rates,2)

In [15]:
def rpf(folder_t, files_t, folder_f, files_f, threshold, exclude=False):
    all_true = predictions_baseline (folder_t, files_t, df_threshold=threshold, exclude_far=exclude)
    all_false = predictions_baseline (folder_f, files_f, df_threshold=threshold, exclude_far=exclude)

    TP = postive_rate(all_true)*len(all_true)
    FN = len(all_true) - TP

    FP = postive_rate(all_false)*len(all_false)
    TN = len(all_false) - FP

    return TP/(TP+FN), TP/(TP+FP), TP/(TP+(FN+FP)*0.5)

In [16]:
folder_positive = "YellowHammer/Training_set_YH_songs/positive/"
folder_negative = 'YellowHammer/Training_set_YH_songs/negative_halfhalf/'

df_folds = pd.read_csv('YellowHammer/Training_set_YH_songs/folds.csv')
df_folds = df_folds.to_numpy()

### Using simple average

In [19]:
neg_train, neg_val, neg_test = negative_folds(folder_negative)

fld_nb = 0
for fold in df_folds:
    fold_train = fold[0:6]
    fold_valid = fold[6:8]
    fold_test = fold[8:10]

    files_train = get_files(folder_positive, fold_train)
    files_validate = get_files(folder_positive, fold_valid)
    files_test = get_files(folder_positive, fold_test)
    
    energy_true = values_db (folder_positive, files_train, exclude_far=False)
    energy_false = values_db (folder_negative, neg_train, exclude_far=False)
    
    threshold = round((np.quantile(energy_true, 0.25) + np.quantile(energy_false, 0.75))/2, 2)

    #print(results(folder_positive, files_test, threshold, exclude=False))
    #print(results(folder_negative, test_neg_fld, threshold, exclude=False))

    #plt.boxplot(energy_true, positions=[1])
    #plt.boxplot(energy_false, positions=[2])
    #plt.show()

    print(threshold, rpf(folder_positive, files_test, folder_negative, neg_test, threshold, exclude=False))


40.23 (0.6826647564469914, 0.5868226600985221, 0.6311258278145695)
40.16 (0.7359180687637161, 0.5970326409495549, 0.6592398427260813)
40.14 (0.6933797909407665, 0.5933214072748957, 0.6394601542416453)
40.14 (0.6883116883116883, 0.596210775606868, 0.6389593908629442)
40.27 (0.649273803119957, 0.6444207154297917, 0.6468381564844587)


# Using precision-recall curves

In [20]:
neg_train, neg_val, neg_test = negative_folds(folder_negative)

fld_nb = 0
for fold in df_folds:
    fold_train = fold[0:6]
    fold_valid = fold[6:8]
    fold_test = fold[8:10]

    files_train = get_files(folder_positive, fold_train)
    files_validate = get_files(folder_positive, fold_valid)
    files_test = get_files(folder_positive, fold_test)
    
    energy_true = values_db (folder_positive, files_train, exclude_far=False)
    energy_false = values_db (folder_negative, neg_train, exclude_far=False)
    
    
    y_true = np.concatenate([[1 for e in energy_true], [0 for e in energy_false]])
    y_scores = np.concatenate([energy_true, energy_false])

    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    f1_score = 2*precision*recall/(precision+recall+1e-10) #1e-10 ESP for div /0


    indx_max = np.argmax(f1_score)
    threshold = round(thresholds[indx_max],2)

    #print(np.round([threshold, recall[indx_max], precision[indx_max], f1_score[indx_max]],2))
    #print("Positive:", results (folder_positive, files_test, threshold, exclude=False))
    #print("Negative:", results (folder_negative, test_neg_fld, threshold, exclude=False))
    print(threshold, rpf(folder_positive, files_test, folder_negative, neg_test, threshold, exclude=False))

    if False:
        plt.plot(thresholds, precision[:-1], label="Precision")
        plt.plot(thresholds, recall[:-1], label="Recall")
        plt.plot(thresholds, f1_score[:-1], label="F1-score")
        plt.vlines(threshold, ymin=0, ymax=np.max(f1_score), colors="black", linestyle='--', label="best F1-score")
        plt.hlines(np.max(f1_score), xmin=np.min(thresholds), xmax=threshold, colors="black", linestyle='--')
        plt.xlabel("Threshold")
        plt.legend()
        plt.show()

        # second 
        plt.plot(recall[:-1], precision[:-1])
        plt.vlines(recall[np.argmax(f1_score)], ymin=0, ymax=precision[np.argmax(f1_score)], colors="black", linestyle='--', label="best F1-score")
        plt.hlines(precision[np.argmax(f1_score)], xmin=0, xmax=recall[np.argmax(f1_score)], colors="black", linestyle='--')
        plt.legend()
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.grid(True)
        plt.show()

48.49 (0.5458452722063037, 0.8513966480446927, 0.6652116979484941)
48.5 (0.6554498902706657, 0.8707482993197279, 0.7479131886477463)
48.49 (0.6250871080139373, 0.870873786407767, 0.727789046653144)
48.49 (0.6008202323991798, 0.8685770750988142, 0.7103030303030304)
48.5 (0.5658956428187197, 0.8877637130801688, 0.6911957950065704)
